In [6]:
import os
import uuid

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.postgres import PostgresSaver

# Short-term Memory

## Base on memory checkpointer

In [11]:
load_dotenv(override=True)

deepseek_model = init_chat_model(model="deepseek:deepseek-v4-flash")
openrouter_model = init_chat_model(model="openrouter:openai/gpt-5.6-luna")

In [11]:
checkpointer = InMemorySaver()
agent = create_agent(model=deepseek_model,
                     checkpointer=checkpointer)

thread_id = uuid.uuid4()

config = RunnableConfig(
    configurable={"thread_id": thread_id}
)

print("\n第一轮对话：")
response1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]},
    config=config  # 传入 config
)
print(f"Agent: {response1['messages'][-1].content}")

print("\n第二轮对话：")
response2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]},
    config=config  # 使用相同的 thread_id
)
print(f"Agent: {response2['messages'][-1].content}")


第一轮对话：
Agent: 你好，张三！很高兴认识你。😊 我是DeepSeek，有什么可以帮你的吗？

第二轮对话：
Agent: 你叫张三呀！刚才你告诉我了，我记得呢～ 😄 有什么需要帮忙的吗？


In [12]:
from rich import print as rprint

latest_state = agent.get_state(config)
rprint(latest_state)

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='我叫张三',
                additional_kwargs={},
                response_metadata={},
                id='8efd80e5-fbc7-4066-b627-fe29adc10725'
            ),
            AIMessage(
                content='你好，张三！很高兴认识你。😊 我是DeepSeek，有什么可以帮你的吗？',
                additional_kwargs={
                    'refusal': None,
                    'reasoning_content': 'We need answer user. User says "我叫张三" Chinese: "My name is Zhang 
San". Need respond friendly. Could introduce self as AI assistant. Keep simple.'
                },
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 57,
                        'prompt_tokens': 85,
                        'total_tokens': 142,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 34,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 85
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402',
                    'id': 'bddd9ed2-b903-4777-81dd-31e3e18300d2',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019fbe51-62dd-7573-b569-3dfbec9cdeb1-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 85,
                    'output_tokens': 57,
                    'total_tokens': 142,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 34}
                }
            ),
            HumanMessage(
                content='我叫什么？',
                additional_kwargs={},
                response_metadata={},
                id='9c50dc98-25a7-4352-9adb-2cf15471988b'
            ),
            AIMessage(
                content='你叫张三呀！刚才你告诉我了，我记得呢～ 😄 有什么需要帮忙的吗？',
                additional_kwargs={
                    'refusal': None,
                    'reasoning_content': 
'我们需要理解用户的问题。用户之前说“我叫张三”，我们回应了“你好，张三！很高兴认识你。”然后用户问“我叫什么？”这看起来
像是一个测试或者玩笑。我们需要根据对话历史来回答。用户已经告诉我们他的名字是张三，所以我们应该回答“你叫张三”。但要
注意，可能用户是在测试我们是否记得，或者是否仔细倾听。另外，也可能用户是在问更广泛的身份，但从上下文看，就是名字。
所以直接回答“你叫张三”即可。还要注意语气友好。可能还需要解释一下为什么我们知道，比如“因为你刚才告诉我你叫张三”。这
样比较自然。\n\n因此，回复应该确认用户的名字，并表明我们记得。可以加上一些友好的语气。'
                },
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 170,
                        'prompt_tokens': 114,
                        'total_tokens': 284,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 148,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 114
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402',
                    'id': 'bd0a88c0-5a6d-4dfe-9b0e-1aab31b76b96',
                    'finish_reason': 'stop',
                    'logprobs': None
          

In [13]:
print("\n第三轮对话：")
response3 = agent.invoke({
    "messages": [HumanMessage("我刚才问了什么问题？")]},
    config=config  # 使用相同的 thread_id
)
print(f"Agent: {response3['messages'][-1].content}")


第三轮对话：
Agent: 你刚才问的是：“我叫什么？” 😄 我回答了你叫张三。需要我再帮你做点什么吗？


In [14]:
from rich import print as rprint

latest_state = agent.get_state(config)
rprint(latest_state)

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='我叫张三',
                additional_kwargs={},
                response_metadata={},
                id='8efd80e5-fbc7-4066-b627-fe29adc10725'
            ),
            AIMessage(
                content='你好，张三！很高兴认识你。😊 我是DeepSeek，有什么可以帮你的吗？',
                additional_kwargs={
                    'refusal': None,
                    'reasoning_content': 'We need answer user. User says "我叫张三" Chinese: "My name is Zhang 
San". Need respond friendly. Could introduce self as AI assistant. Keep simple.'
                },
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 57,
                        'prompt_tokens': 85,
                        'total_tokens': 142,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 34,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 85
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402',
                    'id': 'bddd9ed2-b903-4777-81dd-31e3e18300d2',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019fbe51-62dd-7573-b569-3dfbec9cdeb1-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 85,
                    'output_tokens': 57,
                    'total_tokens': 142,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 34}
                }
            ),
            HumanMessage(
                content='我叫什么？',
                additional_kwargs={},
                response_metadata={},
                id='9c50dc98-25a7-4352-9adb-2cf15471988b'
            ),
            AIMessage(
                content='你叫张三呀！刚才你告诉我了，我记得呢～ 😄 有什么需要帮忙的吗？',
                additional_kwargs={
                    'refusal': None,
                    'reasoning_content': 
'我们需要理解用户的问题。用户之前说“我叫张三”，我们回应了“你好，张三！很高兴认识你。”然后用户问“我叫什么？”这看起来
像是一个测试或者玩笑。我们需要根据对话历史来回答。用户已经告诉我们他的名字是张三，所以我们应该回答“你叫张三”。但要
注意，可能用户是在测试我们是否记得，或者是否仔细倾听。另外，也可能用户是在问更广泛的身份，但从上下文看，就是名字。
所以直接回答“你叫张三”即可。还要注意语气友好。可能还需要解释一下为什么我们知道，比如“因为你刚才告诉我你叫张三”。这
样比较自然。\n\n因此，回复应该确认用户的名字，并表明我们记得。可以加上一些友好的语气。'
                },
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 170,
                        'prompt_tokens': 114,
                        'total_tokens': 284,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 148,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 114
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402',
                    'id': 'bd0a88c0-5a6d-4dfe-9b0e-1aab31b76b96',
                    'finish_reason': 'stop',
                    'logprobs': None
          

In [10]:
config2 = {
    "configurable": {
        "thread_id": "2"
    }
}

response4 = agent.invoke(
    {"messages": [HumanMessage("你还记得我叫什么名字么？")]},
    config=config2
)

print(response4['messages'][-1].content)

抱歉，我记不住你的名字哦！作为一个AI助手，我没有长期记忆，也不会存储个人信息。不过只要你告诉我，我可以随时用你喜欢的称呼来交流～ 😊


## 基于外部存储介质的持久化器

In [12]:
DB_URL = os.getenv("LANGCHAIN_POSTGRES_URL")
if not DB_URL:
    raise RuntimeError("请在 .env 中设置 LANGCHAIN_POSTGRES_URL")
with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    checkpointer.setup()

    agent = create_agent(model=deepseek_model, checkpointer=checkpointer)
    config = RunnableConfig(
        configurable={
            "thread_id": uuid.uuid4()
            # "thread_id": "1f18e352-98c1-6fec-8004-3161c22b2e63"
        }
    )

    print("=" * 30, "-> 第一次调用 <-", "=" * 30)
    response1 = agent.invoke(config=config, input={
        "messages": [
            HumanMessage("你好，我是老王")
        ]
    })
    for msg in response1["messages"]:
        msg.pretty_print()

    print("=" * 30, "-> 第二次调用 <-", "=" * 30)
    response2 = agent.invoke(config=config, input={
        "messages": [
            HumanMessage("你好，我是谁？")
        ]
    })
    for msg in response2["messages"]:
        msg.pretty_print()


============================== -> 第一次调用 <- ==============================
================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

你好，老王！很高兴认识你。有什么我可以帮你的吗？
============================== -> 第二次调用 <- ==============================
================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

你好，老王！很高兴认识你。有什么我可以帮你的吗？
================================ Human Message =================================

你好，我是谁？
================================== Ai Message ==================================

哈哈，根据你刚才的自我介绍，你是老王呀！怎么，这么快就忘了自己是谁吗？😄 还是说，你在考我的记忆力？没问题，我记得清清楚楚。  
如果你在问更深层的“我是谁”，那可能得你自己去探寻了——不过我可以陪你聊聊哲学、人生，或者任何你想聊的话题。
